# Open in Colab — 3D Craniofacial Pipeline

End-to-end Colab entrypoint: **GitHub code → Drive dataset → runtime/GPU check → TotalSegmentator → 2D projections → Drive outputs**.

This runs the existing research prototype; it does not change its scientific limitations.

## 1. Runtime check

Select **Runtime → Change runtime type → GPU** first. Colab does not guarantee a fixed GPU model, so this cell records the hardware actually assigned to the session.

In [ ]:
import os, sys, platform, subprocess
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
   except Exception as e: print('PyTorch not ready yet:', e)
subprocess.run(['nvidia-smi'], check=False)

## 2. Clone and install

The repository pins TotalSegmentator **2.18.0** for this workflow. It requires Python ≥3.9; the upstream package declares PyTorch ≥2.1.2 and nnU-Net v2 as dependencies. citeturn0search0turn0search3

In [ ]:
!rm -rf /content/3D-Craniofacial-Pipeline
!git clone -q https://github.com/trungnb/3D-Craniofacial-Pipeline.git /content/3D-Craniofacial-Pipeline
!python -m pip install -q --upgrade pip
!python -m pip install -q -r /content/3D-Craniofacial-Pipeline/requirements.txt
import totalsegmentator, torch
print('TotalSegmentator:', getattr(totalsegmentator, '__version__', 'installed'))
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## 3. Connect Google Drive

Put CBCT `.nii` / `.nii.gz` files in `My Drive/Pipeline/Data/`. Large files stay in Drive; `/content` is used as temporary scratch space.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, time, json
from datetime import datetime
drive.mount('/content/drive')
REPO = Path('/content/3D-Craniofacial-Pipeline')
DRIVE_ROOT = Path('/content/drive/MyDrive/Pipeline')
DATA_DIR = DRIVE_ROOT / 'Data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Dataset folder:', DATA_DIR)

## 4. Choose dataset and scans

**Recommended first test:** one NIfTI scan.

Public CBCT options include **ToothFairy3** (532 NIfTI CBCT volumes, 77 classes; registration required) and **ToothFairy2** (official page requires an account to download). citeturn2search0turn3view0

For a small smoke test, a public CTooth dataset is also available in NIfTI format. citeturn2search7turn2search10

In [ ]:
from ipywidgets import SelectMultiple, Layout
from IPython.display import display
nii_files = sorted(p for p in DATA_DIR.iterdir() if p.is_file() and (p.name.endswith('.nii') or p.name.endswith('.nii.gz')))
if not nii_files:
    raise FileNotFoundError(f'No NIfTI files in {DATA_DIR}. Add a .nii/.nii.gz file, then rerun this cell.')
selector = SelectMultiple(options=[(p.name, str(p)) for p in nii_files], value=(str(nii_files[0]),), description='Scans:', rows=min(12,max(3,len(nii_files))), layout=Layout(width='90%'))
display(selector)

In [ ]:
selected = [Path(p) for p in selector.value]
if not selected: raise ValueError('Select at least one scan.')
RUN_ID = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
RUN_ROOT = DRIVE_ROOT / 'runs' / RUN_ID
LOCAL_INPUT = Path('/content/cbct_pipeline/input')
LOCAL_OUTPUT = Path('/content/cbct_pipeline/output')
LOCAL_INPUT.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)
for src in selected: shutil.copy2(src, LOCAL_INPUT / src.name)
print('Selected:', [p.name for p in selected])
print('Run output:', RUN_ROOT)

## 5. Run segmentation + projections

The original prototype uses `craniofacial_structures` and `teeth`, with `fast=False`. This runner preserves those choices rather than silently changing the model/resolution.

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from totalsegmentator.python_api import totalsegmentator

def make_projections(mask_path, out_dir):
    img = nib.load(str(mask_path)); data = img.get_fdata(); mask = (data > 0).astype(np.uint8)
    out_dir.mkdir(parents=True, exist_ok=True)
    plt.imsave(out_dir/'axial_projection.png', np.max(mask, axis=2), cmap='gray')
    plt.imsave(out_dir/'coronal_projection.png', np.max(mask, axis=1), cmap='gray')
    plt.imsave(out_dir/'sagital_projection.png', np.max(mask, axis=0), cmap='gray')

summary=[]
for src in sorted(LOCAL_INPUT.iterdir()):
    if not (src.name.endswith('.nii') or src.name.endswith('.nii.gz')): continue
    name=src.name.replace('.nii.gz','').replace('.nii','')
    seg=LOCAL_OUTPUT/'Volumes'/name; proj=LOCAL_OUTPUT/'Projections'/name
    seg.mkdir(parents=True, exist_ok=True)
    t0=time.time(); print(f'=== {name} ===')
    totalsegmentator(input=str(src), output=str(seg), task='craniofacial_structures', output_type='niftis', fast=False)
    totalsegmentator(input=str(src), output=str(seg), task='teeth', output_type='niftis', fast=False)
    masks=list(sorted(seg.glob('*.nii*')))
    for mask in masks: make_projections(mask, proj/mask.name.replace('.nii.gz','').replace('.nii',''))
    elapsed=time.time()-t0; summary.append({'scan':name,'seconds':elapsed,'mask_files':len(masks)})
    print(f'Completed in {elapsed:.1f}s; masks={len(masks)}')

## 6. Save results and runtime manifest

In [ ]:
import pandas as pd
RUN_ROOT.mkdir(parents=True, exist_ok=False)
for folder in ['Volumes','Projections']:
    src=LOCAL_OUTPUT/folder
    if src.exists(): shutil.copytree(src, RUN_ROOT/folder, dirs_exist_ok=True)
torch=__import__('torch')
manifest={
  'run_id':RUN_ID, 'selected_scans':[p.name for p in selected],
  'python':sys.version, 'pytorch':torch.__version__,
  'cuda_available':bool(torch.cuda.is_available()),
  'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
  'totalsegmentator':'2.18.0', 'tasks':['craniofacial_structures','teeth'], 'fast':False,
  'timestamp_utc':datetime.utcnow().isoformat()+'Z'
}
(RUN_ROOT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
pd.DataFrame(summary).to_csv(RUN_ROOT/'run_summary.csv',index=False)
print(json.dumps(manifest,indent=2)); print('Saved:',RUN_ROOT)

## 7. Quick visual check

In [ ]:
from IPython.display import Image, display
first=selected[0].name.replace('.nii.gz','').replace('.nii','')
pngs=sorted((RUN_ROOT/'Projections'/first).glob('*/*.png'))
for p in pngs[:6]:
    print(p.relative_to(RUN_ROOT)); display(Image(filename=str(p),width=360))

### Reproducibility

Colab's GPU type and runtime limits vary over time; the assigned hardware is therefore recorded in `run_manifest.json`. Free Colab runtimes are not guaranteed and can terminate. citeturn1search0

For this repository, TotalSegmentator 2.18.0 is pinned so the execution environment is explicit. citeturn0search0turn0search3